In [1]:
# ==============================
# 🔧 INSTALAÇÃO
# ==============================
# !pip install -q sentence-transformers transformers

# ==============================
# 📌 IMPORTS
# ==============================
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re

# ==============================
# ✍️ DADOS
# ==============================

RESPOSTA = """
Maria não cometeu ato ilícito, pois a concessão de aposentadoria é ato complexo
que depende da análise do tribunal de contas.

Além disso, o contraditório e a ampla defesa são garantidos no tribunal de contas,
exceto na análise inicial de aposentadoria.
"""

# ==============================
# 🧠 MODELOS
# ==============================

sbert_model = SentenceTransformer('stjiris/bert-large-portuguese-cased-legal-mlm-sts-v1.0')

device = 0 if torch.cuda.is_available() else -1
nli_model = pipeline(
    "text-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)

# ==============================
# 🔍 FUNÇÕES DE AVALIAÇÃO
# ==============================

def calcular_sbert(texto1, texto2):
    emb1 = sbert_model.encode(texto1, convert_to_tensor=True)
    emb2 = sbert_model.encode(texto2, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()


def calcular_nli(resposta, referencia):
    result = nli_model(f"{referencia} </s> {resposta}")
    label = result[0]['label'].upper()
    score = result[0]['score']

    if label == "ENTAILMENT":
        return score
    elif label == "NEUTRAL":
        return 0.5 * score
    else:  # CONTRADICTION
        return 1 - score


def avaliar_semantico(resposta, referencia):
    sbert = calcular_sbert(resposta, referencia)
    nli = calcular_nli(resposta, referencia)

    if nli < 0.2:
        return 0

    return (0.7 * sbert) + (0.3 * nli)


def normalizar(texto):
    texto = texto.lower()
    texto = texto.replace("artigo", "art")
    texto = texto.replace("nº", "")
    return texto


def verificar_legislacao(resposta, termos):
    texto = normalizar(resposta)

    for termo in termos:
        if termo in texto:
            return True

    return False


# ==============================
# 💬 FUNÇÕES DE EXPLICAÇÃO
# ==============================

def explicar_criterio_semantico(nome, score, nota, peso, referencia):
    linhas = [f"🔹 {nome}"]

    if score == 0:
        linhas.append("  ❌ Contradição detectada com a referência — score zerado.")
    elif score >= 0.85:
        linhas.append(f"  ✅ Alta aderência semântica ao critério (score: {score:.2f}).")
    elif score >= 0.65:
        linhas.append(f"  ⚠️ Aderência semântica moderada ao critério (score: {score:.2f}).")
    else:
        linhas.append(f"  ❌ Baixa aderência semântica ao critério (score: {score:.2f}).")

    linhas.append(f"  📌 Referência: \"{referencia[:80]}{'...' if len(referencia) > 80 else ''}\"")
    linhas.append(f"  📊 Nota: {round(nota, 3)} / {peso}")
    return "\n".join(linhas)


def explicar_criterio_legislacao(nome, match, termos, nota, peso):
    linhas = [f"🔹 {nome}"]

    termos_fmt = ", ".join(termos)
    if match:
        linhas.append(f"  ✅ Referência legal identificada na resposta (esperado: {termos_fmt}).")
    else:
        linhas.append(f"  ❌ Nenhum dos termos legais exigidos foi encontrado na resposta (esperado: {termos_fmt}).")

    linhas.append(f"  📊 Nota: {round(nota, 3)} / {peso}")
    return "\n".join(linhas)


def gerar_resumo_final(nota_total, peso_total, detalhes):
    percentual = (nota_total / peso_total) * 100

    if percentual >= 85:
        nivel = "Excelente"
    elif percentual >= 65:
        nivel = "Satisfatório"
    elif percentual >= 40:
        nivel = "Parcial"
    else:
        nivel = "Insuficiente"

    criterios_ok  = [d["nome"] for d in detalhes if d["score"] >= 0.65 or (d["tipo"] == "legislacao" and d["score"] == 1)]
    criterios_nok = [d["nome"] for d in detalhes if d["score"] < 0.65]

    linhas = []
    if criterios_ok:
        linhas.append(f"  ✅ Critérios bem atendidos: {', '.join(criterios_ok)}.")
    if criterios_nok:
        linhas.append(f"  ⚠️ Critérios com baixo desempenho: {', '.join(criterios_nok)}.")

    linhas.append(f"\n  📝 Nota final: {round(nota_total, 2)} / {peso_total} ({percentual:.0f}%) — {nivel}.")
    return "\n".join(linhas)


# ==============================
# 🎯 CRITÉRIOS (HARDCODE)
# ==============================

criterios = [
    {
        "nome": "A1 - ato complexo",
        "tipo": "semantico",
        "referencia": "A concessão de aposentadoria configura ato complexo que só estará perfeito após pronunciamento da Corte de Contas",
        "peso": 0.5
    },
    {
        "nome": "A2 - base legal",
        "tipo": "legislacao",
        "termos": ["71", "sumula 3"],
        "peso": 0.1
    },
    {
        "nome": "B1 - contraditorio com excecao",
        "tipo": "semantico",
        "referencia": "O contraditório e a ampla defesa são assegurados no Tribunal de Contas da União quando da decisão puder resultar anulação ou revogação de ato administrativo, excetuada a apreciação de legalidade do ato de concessão inicial de aposentadoria, reforma e pensão",
        "peso": 0.55
    },
    {
        "nome": "B2 - sumula 3",
        "tipo": "legislacao",
        "termos": ["sumula 3"],
        "peso": 0.1
    }
]

# ==============================
# 🧮 AVALIAÇÃO
# ==============================

nota_total = 0
peso_total = sum(c["peso"] for c in criterios)
detalhes = []
explicacoes = []

print("\n==============================")
print("📊 AVALIAÇÃO POR CRITÉRIO")
print("==============================\n")

for c in criterios:

    if c["tipo"] == "semantico":
        score = avaliar_semantico(RESPOSTA, c["referencia"])
        nota = score * c["peso"]
        explicacoes.append(explicar_criterio_semantico(c["nome"], score, nota, c["peso"], c["referencia"]))

    else:
        match = verificar_legislacao(RESPOSTA, c["termos"])
        nota = c["peso"] if match else 0
        score = 1 if match else 0
        explicacoes.append(explicar_criterio_legislacao(c["nome"], match, c["termos"], nota, c["peso"]))

    nota_total += nota
    detalhes.append({"nome": c["nome"], "tipo": c["tipo"], "score": score})

    print(f"{c['nome']}")
    print(f"Score: {round(score, 3)}")
    print(f"Nota: {round(nota, 3)} / {c['peso']}\n")

# ==============================
# 💬 EXPLICAÇÃO
# ==============================

print("==============================")
print("💬 EXPLICAÇÃO DA AVALIAÇÃO")
print("==============================\n")

for exp in explicacoes:
    print(exp)
    print()

print("==============================")
print("📋 RESUMO FINAL")
print("==============================")
print(gerar_resumo_final(nota_total, peso_total, detalhes))

print("\n==============================")
print(f"🎯 NOTA FINAL: {round(nota_total, 2)}")
print("==============================")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

...

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]


📊 AVALIAÇÃO POR CRITÉRIO

A1 - ato complexo
Score: 0.562
Nota: 0.281 / 0.5

A2 - base legal
Score: 0
Nota: 0 / 0.1

B1 - contraditorio com excecao
Score: 0.762
Nota: 0.419 / 0.55

B2 - sumula 3
Score: 0
Nota: 0 / 0.1

💬 EXPLICAÇÃO DA AVALIAÇÃO

🔹 A1 - ato complexo
  ❌ Baixa aderência semântica ao critério (score: 0.56).
  📌 Referência: "A concessão de aposentadoria configura ato complexo que só estará perfeito após ..."
  📊 Nota: 0.281 / 0.5

🔹 A2 - base legal
  ❌ Nenhum dos termos legais exigidos foi encontrado na resposta (esperado: 71, sumula 3).
  📊 Nota: 0 / 0.1

🔹 B1 - contraditorio com excecao
  ⚠️ Aderência semântica moderada ao critério (score: 0.76).
  📌 Referência: "O contraditório e a ampla defesa são assegurados no Tribunal de Contas da União ..."
  📊 Nota: 0.419 / 0.55

🔹 B2 - sumula 3
  ❌ Nenhum dos termos legais exigidos foi encontrado na resposta (esperado: sumula 3).
  📊 Nota: 0 / 0.1

📋 RESUMO FINAL
  ✅ Critérios bem atendidos: B1 - contraditorio com excecao.
  ⚠️ C